# 0825_peace_020_type_expert_preprocessing_correlation_residual

004 단일 타입별 Walk-forward를 대조군으로 고정하고 **안전 전처리 + 안정 상관 Residual**만 추가합니다.

- 상수·완전중복 판정과 상관/회귀 학습은 각 타입 Train에서만 수행합니다.
- 행 삭제, 이상치 삭제, 준상수 제거, label 기반 선택은 하지 않습니다.
- Walk-forward와 Validation만 평가하며 마지막 20% Test는 봉인합니다.

## 1. 설정, 경로 탐색과 실행 로그

노트북을 저장소 루트 또는 `notebooks/`에서 실행해도 같은 원본 파일과 로그 경로를 사용합니다.


In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_020_type_expert_preprocessing_correlation_residual"
USE_RESIDUAL_FEATURES = True
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))

2026-08-25 20:37:28,095 | INFO | experiment=0825_peace_020_type_expert_preprocessing_correlation_residual


2026-08-25 20:37:28,096 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 20:37:28,096 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 20:37:28,096 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 20:37:28,097 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 20:37:28,097 | INFO | log_file=docs/peace/0825_peace_020_type_expert_preprocessing_correlation_residual.log


log saved to: docs/peace/0825_peace_020_type_expert_preprocessing_correlation_residual.log


## 2. 원본 데이터와 매핑 검증

원본 행과 파일은 수정하지 않습니다.

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 20:37:32,505 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 후보 피처와 Train-only 처리

Mapping 후보에서 Train 상수·완전중복을 제거합니다. 020은 시간 3블록 모두에서 안정적인 센서 관계의 Residual만 추가합니다.

In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 20:37:32,515 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 시간순 Train/Validation/Test 분할

전체 행의 누적 비율에 가장 가까운 timestamp 그룹 끝을 경계로 사용합니다. 같은 timestamp 그룹은 서로 다른 구간에 들어가지 않습니다.

- 0~70%: Train
- 70~80%: Validation
- 80~100%: 최종 Test


In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": np.nan,
            "positive_rate_pct": np.nan,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "test_status": "sealed",
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)

,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940.0,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357.0,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,NaN,NaN,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test      False
threshold_selected_on_test     False
test_status                   sealed
Name: evaluation_policy, dtype: object

2026-08-25 20:37:32,878 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940.0, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357.0, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': nan, 'positive_rate_pct': nan, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 20:37:32,879 | INFO | test_policy model_selection=False threshold=0.50


## 5. Peace 실험과 동일한 평가 지표

PR-AUC, ROC-AUC, Accuracy, Precision, Recall, F1, TP/FN/FP/TN, False Call Reduction을 계산합니다. Threshold 0.5는 베이스라인 비교용이며 운영 임계값이 아닙니다.


In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def select_safe_features(frame, candidates):
    constants = [c for c in candidates if frame[c].nunique(dropna=False) <= 1]
    selected, duplicates, signatures = [], {}, {}
    for column in (c for c in candidates if c not in constants):
        series = frame[column]
        hashed = pd.util.hash_pandas_object(series, index=False, categorize=True).to_numpy(np.uint64)
        signature = (str(series.dtype), hashlib.sha256(hashed.tobytes()).hexdigest())
        match = next((r for r in signatures.get(signature, [])
                      if series.reset_index(drop=True).equals(frame[r].reset_index(drop=True))), None)
        if match is None:
            selected.append(column); signatures.setdefault(signature, []).append(column)
        else:
            duplicates[column] = match
    assert len(selected) + len(constants) + len(duplicates) == len(candidates)
    return selected, constants, duplicates

def chronological_blocks(frame):
    sizes = frame.groupby(TIME_COLUMN, sort=True).size()
    cumulative, times = sizes.cumsum().to_numpy(), sizes.index
    b1, b2 = [times[int(np.searchsorted(cumulative, len(frame)*f, side="left"))]
              for f in (1/3, 2/3)]
    blocks = [frame.loc[frame[TIME_COLUMN] <= b1],
              frame.loc[(frame[TIME_COLUMN] > b1) & (frame[TIME_COLUMN] <= b2)],
              frame.loc[frame[TIME_COLUMN] > b2]]
    assert all(len(block) for block in blocks) and sum(map(len, blocks)) == len(frame)
    return blocks

def fit_feature_recipe(frame, candidates):
    selected, constants, duplicates = select_safe_features(frame, candidates)
    specs = []
    if USE_RESIDUAL_FEATURES:
        sensors = [c for c in selected if c.startswith("inspection_feat")]
        matrices = [b[sensors].corr(method="spearman") for b in chronological_blocks(frame)]
        pairs = []
        for i, left in enumerate(sensors):
            for right in sensors[i+1:]:
                rhos = np.array([m.loc[left, right] for m in matrices], dtype=float)
                if (np.isfinite(rhos).all() and np.all(np.abs(rhos) >= 0.90)
                        and np.all(np.sign(rhos) == np.sign(rhos[0]))):
                    pairs.append((float(np.min(np.abs(rhos))), left, right, rhos))
        used = set()
        for score, left, right, rhos in sorted(pairs, key=lambda x: (-x[0], x[1], x[2])):
            if left in used or right in used: continue
            x, y = frame[left].to_numpy(float), frame[right].to_numpy(float)
            variance = float(np.var(x))
            if not np.isfinite(variance) or variance <= 0: continue
            slope = float(np.mean((x-x.mean())*(y-y.mean())) / variance)
            k = len(specs)
            specs.append({"left": left, "right": right, "block_rhos": rhos.tolist(),
                          "min_abs_rho": score, "slope": slope,
                          "intercept": float(y.mean()-slope*x.mean()),
                          "residual": f"corr_resid_{k:02d}",
                          "absolute": f"corr_abs_resid_{k:02d}"})
            used.update((left, right))
            if len(specs) == 10: break
    model_columns = list(selected)
    for spec in specs: model_columns += [spec["residual"], spec["absolute"]]
    return {"selected": selected, "constants": constants, "duplicates": duplicates,
            "residuals": specs, "model_columns": model_columns}

def transform_recipe(frame, recipe):
    result = frame[recipe["selected"]].copy()
    for spec in recipe["residuals"]:
        x, y = frame[spec["left"]].to_numpy(float), frame[spec["right"]].to_numpy(float)
        residual = (y-(spec["slope"]*x+spec["intercept"]))/np.sqrt(spec["slope"]**2+1)
        result[spec["residual"]] = residual.astype(np.float32)
        result[spec["absolute"]] = np.abs(residual).astype(np.float32)
    assert list(result) == recipe["model_columns"]
    return result

2026-08-25 20:37:32,910 | INFO | threshold_selector_unit_test=PASS


## 6. 3-Fold Expanding Walk-forward 검증

첫 70% 개발 구간 안에서 Train을 누적 확장합니다. 각 Fold의 Calibration에서 임계값을 선택하고, 그 임계값을 바로 다음 미래 Evaluation에 고정 적용합니다.

| Fold | Train | Calibration | Evaluation |
|---|---:|---:|---:|
| Fold 1 | 0~30% | 30~40% | 40~50% |
| Fold 2 | 0~40% | 40~50% | 50~60% |
| Fold 3 | 0~50% | 50~60% | 60~70% |

Calibration과 Evaluation은 모델 학습에 사용하지 않으며, Evaluation은 임계값 선택에도 사용하지 않습니다.

In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 20:37:33,564 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

In [7]:
def fit_type_experts_for_fold(train_frame, calibration_frame, evaluation_frame, fold_name):
    cal_prob = pd.Series(np.nan, index=calibration_frame.index, dtype=float)
    eval_prob = pd.Series(np.nan, index=evaluation_frame.index, dtype=float)
    audits = []
    for inspection_type in inspection_types:
        candidates = feature_columns_by_type[inspection_type]
        train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type]
        cal = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        ev = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        assert train[TARGET].nunique() == 2 and cal[TARGET].nunique() == 2
        recipe = fit_feature_recipe(train, candidates)
        train_x, cal_x, ev_x = [transform_recipe(x, recipe) for x in (train, cal, ev)]
        prep = make_preprocessor(recipe["model_columns"])
        X_train = prep.fit_transform(train_x); X_cal = prep.transform(cal_x); X_eval = prep.transform(ev_x)
        model = XGBClassifier(**XGB_PARAMS).fit(X_train, train[TARGET].astype("int8"), verbose=False)
        cal_prob.loc[cal.index] = model.predict_proba(X_cal)[:, 1]
        eval_prob.loc[ev.index] = model.predict_proba(X_eval)[:, 1]
        row = {"fold": fold_name, "inspection_type": inspection_type,
               "train_rows": len(train), "candidate_features": len(candidates),
               "constant_removed": len(recipe["constants"]),
               "duplicate_removed": len(recipe["duplicates"]),
               "residual_pairs": len(recipe["residuals"]),
               "model_features": len(recipe["model_columns"]), "encoded_features": X_train.shape[1]}
        audits.append(row)
        logger.info("fold_recipe summary=%s constants=%s duplicates=%s residuals=%s",
                    row, recipe["constants"], recipe["duplicates"], recipe["residuals"])
        del model, prep, X_train, X_cal, X_eval, train_x, cal_x, ev_x; gc.collect()
    assert cal_prob.notna().all() and eval_prob.notna().all()
    return cal_prob, eval_prob, audits

walk_forward_threshold_rows, walk_forward_metric_rows = [], []
walk_forward_type_evaluation_rows, walk_forward_training_rows = [], []
for spec in WALK_FORWARD_SPECS:
    fold, segments = spec["fold"], walk_forward_segments[spec["fold"]]
    cal, ev = segments["calibration"], segments["evaluation"]
    cal_prob, eval_prob, audits = fit_type_experts_for_fold(segments["train"], cal, ev, fold)
    walk_forward_training_rows += audits
    global_selection = select_threshold(cal[TARGET], cal_prob, MIN_RECALL)
    walk_forward_threshold_rows.append({"fold": fold, "scope": "global", **global_selection})
    type_prediction = pd.Series(np.nan, index=ev.index, dtype=float)
    for inspection_type in inspection_types:
        tc = cal.loc[cal[TYPE_COLUMN] == inspection_type]
        selection = select_threshold(tc[TARGET], cal_prob.loc[tc.index], MIN_RECALL)
        walk_forward_threshold_rows.append({"fold": fold, "scope": f"type_{inspection_type}", **selection})
        te = ev.loc[ev[TYPE_COLUMN] == inspection_type]; p = eval_prob.loc[te.index]
        pred = (p >= selection["threshold"]).astype("int8"); type_prediction.loc[te.index] = pred
        metrics = evaluate_predictions(te[TARGET], pred, p)
        metrics.update({"fold": fold, "inspection_type": inspection_type,
                        "threshold": selection["threshold"]})
        walk_forward_type_evaluation_rows.append(metrics)
    strategies = {"fixed_0.5": evaluate_probabilities(ev[TARGET], eval_prob, DECISION_THRESHOLD),
                  "global_threshold": evaluate_probabilities(ev[TARGET], eval_prob, global_selection["threshold"]),
                  "type_specific_thresholds": evaluate_predictions(ev[TARGET], type_prediction, eval_prob)}
    walk_forward_metric_rows += [{"fold": fold, "strategy": key, **value}
                                 for key, value in strategies.items()]
    logger.info("walk_forward_fold_done fold=%s metrics=%s", fold, strategies)
walk_forward_threshold_summary = pd.DataFrame(walk_forward_threshold_rows).set_index(["fold", "scope"])
walk_forward_evaluation_metrics = pd.DataFrame(walk_forward_metric_rows).set_index(["fold", "strategy"])
walk_forward_type_evaluation = pd.DataFrame(walk_forward_type_evaluation_rows).set_index(["fold", "inspection_type"])
walk_forward_training_summary = pd.DataFrame(walk_forward_training_rows).set_index(["fold", "inspection_type"])

2026-08-25 20:37:34,037 | INFO | fold_recipe summary={'fold': 'fold_1', 'inspection_type': 0, 'train_rows': 28277, 'candidate_features': 48, 'constant_removed': 14, 'duplicate_removed': 0, 'residual_pairs': 7, 'model_features': 48, 'encoded_features': 80} constants=['meta_feat3', 'inspection_feat1', 'inspection_feat2', 'inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat19', 'inspection_feat22', 'inspection_feat30', 'inspection_feat31', 'inspection_feat47', 'inspection_feat48', 'inspection_feat57', 'inspection_feat58'] duplicates={} residuals=[{'left': 'inspection_feat27', 'right': 'inspection_feat28', 'block_rhos': [0.9946444845299715, 0.9972857532379601, 0.9950043685235155], 'min_abs_rho': 0.9946444845299715, 'slope': 0.023552202944044053, 'intercept': 0.16793563516071103, 'residual': 'corr_resid_00', 'absolute': 'corr_abs_resid_00'}, {'left': 'inspection_feat44', 'right': 'inspection_feat45', 'block_rhos': [0.9909933891623742, 0.9949790651508374, 0.9959965

2026-08-25 20:37:34,461 | INFO | fold_recipe summary={'fold': 'fold_1', 'inspection_type': 1, 'train_rows': 22698, 'candidate_features': 56, 'constant_removed': 28, 'duplicate_removed': 2, 'residual_pairs': 5, 'model_features': 36, 'encoded_features': 86} constants=['inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat20', 'inspection_feat21', 'inspection_feat22', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection_feat58', 'inspection_feat59', 'inspection_feat71', 'inspection_feat72', 'inspection_feat78', 'inspection_feat93', 'inspection_feat94', 'inspection_feat95', 'inspection_feat96'] duplicates={'inspection_feat32': 'inspection_feat5', 'inspection_feat33': 'inspection_feat6'} residuals=[{'left': 'inspection_feat9', 'right': 'i

2026-08-25 20:37:35,191 | INFO | fold_recipe summary={'fold': 'fold_1', 'inspection_type': 2, 'train_rows': 42288, 'candidate_features': 69, 'constant_removed': 45, 'duplicate_removed': 0, 'residual_pairs': 1, 'model_features': 26, 'encoded_features': 71} constants=['inspection_feat2', 'inspection_feat6', 'inspection_feat13', 'inspection_feat16', 'inspection_feat17', 'inspection_feat21', 'inspection_feat23', 'inspection_feat24', 'inspection_feat25', 'inspection_feat26', 'inspection_feat27', 'inspection_feat29', 'inspection_feat30', 'inspection_feat31', 'inspection_feat32', 'inspection_feat33', 'inspection_feat34', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat48', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection_feat57', 'inspection_feat59', 'inspection_feat63', 'inspection_feat64', 

2026-08-25 20:37:35,767 | INFO | fold_recipe summary={'fold': 'fold_1', 'inspection_type': 3, 'train_rows': 37264, 'candidate_features': 69, 'constant_removed': 48, 'duplicate_removed': 0, 'residual_pairs': 0, 'model_features': 21, 'encoded_features': 59} constants=['meta_feat3', 'inspection_feat2', 'inspection_feat6', 'inspection_feat7', 'inspection_feat13', 'inspection_feat16', 'inspection_feat18', 'inspection_feat19', 'inspection_feat21', 'inspection_feat23', 'inspection_feat24', 'inspection_feat25', 'inspection_feat26', 'inspection_feat27', 'inspection_feat29', 'inspection_feat30', 'inspection_feat31', 'inspection_feat32', 'inspection_feat33', 'inspection_feat34', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat48', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection_feat57', 'inspect

2026-08-25 20:37:35,847 | INFO | fold_recipe summary={'fold': 'fold_1', 'inspection_type': 4, 'train_rows': 1610, 'candidate_features': 25, 'constant_removed': 8, 'duplicate_removed': 2, 'residual_pairs': 0, 'model_features': 15, 'encoded_features': 37} constants=['inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat20', 'inspection_feat21', 'inspection_feat22', 'inspection_feat23', 'inspection_feat48'] duplicates={'inspection_feat32': 'inspection_feat5', 'inspection_feat33': 'inspection_feat6'} residuals=[]


...

2026-08-25 20:37:40,563 | INFO | fold_recipe summary={'fold': 'fold_3', 'inspection_type': 1, 'train_rows': 29184, 'candidate_features': 56, 'constant_removed': 28, 'duplicate_removed': 2, 'residual_pairs': 6, 'model_features': 38, 'encoded_features': 93} constants=['inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat20', 'inspection_feat21', 'inspection_feat22', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection_feat58', 'inspection_feat59', 'inspection_feat71', 'inspection_feat72', 'inspection_feat78', 'inspection_feat93', 'inspection_feat94', 'inspection_feat95', 'inspection_feat96'] duplicates={'inspection_feat32': 'inspection_feat5', 'inspection_feat33': 'inspection_feat6'} residuals=[{'left': 'inspection_feat9', 'right': 'i

2026-08-25 20:37:41,461 | INFO | fold_recipe summary={'fold': 'fold_3', 'inspection_type': 2, 'train_rows': 77700, 'candidate_features': 69, 'constant_removed': 45, 'duplicate_removed': 0, 'residual_pairs': 1, 'model_features': 26, 'encoded_features': 72} constants=['inspection_feat2', 'inspection_feat6', 'inspection_feat13', 'inspection_feat16', 'inspection_feat17', 'inspection_feat21', 'inspection_feat23', 'inspection_feat24', 'inspection_feat25', 'inspection_feat26', 'inspection_feat27', 'inspection_feat29', 'inspection_feat30', 'inspection_feat31', 'inspection_feat32', 'inspection_feat33', 'inspection_feat34', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat48', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection_feat57', 'inspection_feat59', 'inspection_feat63', 'inspection_feat64', 

2026-08-25 20:37:42,524 | INFO | fold_recipe summary={'fold': 'fold_3', 'inspection_type': 3, 'train_rows': 67320, 'candidate_features': 69, 'constant_removed': 47, 'duplicate_removed': 0, 'residual_pairs': 0, 'model_features': 22, 'encoded_features': 61} constants=['meta_feat3', 'inspection_feat2', 'inspection_feat6', 'inspection_feat13', 'inspection_feat16', 'inspection_feat18', 'inspection_feat19', 'inspection_feat21', 'inspection_feat23', 'inspection_feat24', 'inspection_feat25', 'inspection_feat26', 'inspection_feat27', 'inspection_feat29', 'inspection_feat30', 'inspection_feat31', 'inspection_feat32', 'inspection_feat33', 'inspection_feat34', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat48', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection_feat57', 'inspection_feat59', 'inspec

2026-08-25 20:37:42,605 | INFO | fold_recipe summary={'fold': 'fold_3', 'inspection_type': 4, 'train_rows': 2771, 'candidate_features': 25, 'constant_removed': 8, 'duplicate_removed': 2, 'residual_pairs': 0, 'model_features': 15, 'encoded_features': 40} constants=['inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat20', 'inspection_feat21', 'inspection_feat22', 'inspection_feat23', 'inspection_feat48'] duplicates={'inspection_feat32': 'inspection_feat5', 'inspection_feat33': 'inspection_feat6'} residuals=[]


2026-08-25 20:37:42,873 | INFO | walk_forward_fold_done fold=fold_3 metrics={'fixed_0.5': {'rows': 43853, 'positive_samples': 39, 'tn': 43764, 'fp': 50, 'fn': 36, 'tp': 3, 'accuracy': 0.9980389026976489, 'precision': 0.05660377358490566, 'recall': 0.07692307692307693, 'false_call_reduction': 0.9988588122517916, 'f1': 0.06521739130434782, 'roc_auc': 0.9227887000174397, 'pr_auc': 0.02766778255926858}, 'global_threshold': {'rows': 43853, 'positive_samples': 39, 'tn': 4218, 'fp': 39596, 'fn': 0, 'tp': 39, 'accuracy': 0.0970743164663763, 'precision': 0.000983978806610319, 'recall': 1.0, 'false_call_reduction': 0.09627059843885516, 'f1': 0.001966023088168574, 'roc_auc': 0.9227887000174397, 'pr_auc': 0.02766778255926858}, 'type_specific_thresholds': {'rows': 43853, 'positive_samples': 39, 'tn': 7326, 'fp': 36488, 'fn': 0, 'tp': 39, 'accuracy': 0.1679474608350626, 'precision': 0.001067703342732773, 'recall': 1.0, 'false_call_reduction': 0.16720682886748528, 'f1': 0.0021331291363561778, 'roc_au

## 7. Walk-forward 미래 Evaluation 결과

공통·타입별 임계값은 각 Fold의 Calibration에서만 선택됐습니다. 아래 지표는 임계값 선택에 사용하지 않은 바로 다음 미래 Evaluation 결과입니다.

In [8]:
display(
    walk_forward_threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]
    ]
)
display(
    walk_forward_evaluation_metrics[
        [
            "positive_samples",
            "pr_auc",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(
    walk_forward_type_evaluation[
        [
            "threshold",
            "positive_samples",
            "pr_auc",
            "recall",
            "false_call_reduction",
            "tp",
            "fn",
        ]
    ]
)

walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index()
    .groupby("strategy")
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"),
        total_fn=("fn", "sum"),
    )
)
display(walk_forward_strategy_summary)
display(walk_forward_training_summary)
logger.info(
    "walk_forward_strategy_summary=%s",
    walk_forward_strategy_summary.to_dict(orient="index"),
)


threshold  positive_samples    recall  false_call_reduction  \
fold   scope                                                                 
fold_1 global   0.000028               200  0.990000              0.014299   
       type_0   0.002325                11  1.000000              0.375968   
       type_1   0.001276                20  1.000000              0.223233   
       type_2   0.000020                92  1.000000              0.008988   
       type_3   0.000354                73  1.000000              0.173358   
       type_4   0.002461                 4  1.000000              0.000000   
fold_2 global   0.000903               326  0.990798              0.399597   
       type_0   0.000887                50  1.000000              0.536457   
       type_1   0.000405               186  0.994624              0.150905   
       type_2   0.000937                49  1.000000              0.483320   
       type_3   0.006896                39  1.000000              0.611617   
       type_4   0.003340                 2  1.000000              0.000000   
fold_3 global   0.000083               152  0.993421              0.109299   
       type_0   0.000050                14  1.000000              0.028313   
       type_1   0.000490                80  1.000000              0.088003   
       type_2   0.000083                32  1.000000              0.014939   
       type_3   0.000198                23  1.000000              0.490784   
       type_4   0.003684                 3  1.000000              0.000000   

                tp  fn  
fold   scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
fold   strategy                                                          
fold_1 fixed_0.5                              326  0.135803   0.233161   
       global_threshold                       326  0.135803   0.007472   
       type_specific_thresholds               326  0.135803   0.008815   
fold_2 fixed_0.5                              152  0.025417   0.018692   
       global_threshold                       152  0.025417   0.005876   
       type_specific_thresholds               152  0.025417   0.007104   
fold_3 fixed_0.5                               39  0.027668   0.056604   
       global_threshold                        39  0.027668   0.000984   
       type_specific_thresholds                39  0.027668   0.001068   

                                   recall  false_call_reduction        f1  \
fold   strategy                                                             
fold_1 fixed_0.5                 0.138037              0.996614  0.173410   
       global_threshold          1.000000              0.009379  0.014833   
       type_specific_thresholds  0.981595              0.176854  0.017472   
fold_2 fixed_0.5                 0.026316              0.995231  0.021858   
       global_threshold          0.901316              0.473601  0.011675   
       type_specific_thresholds  0.796053              0.615942  0.014082   
fold_3 fixed_0.5                 0.076923              0.998859  0.065217   
       global_threshold          1.000000              0.096271  0.001966   
       type_specific_thresholds  1.000000              0.167207  0.002133   

                                  tp   fn     fp     tn  
fold   strategy                                          
fold_1 fixed_0.5                  45  281    148  43566  
       global_threshold          326    0  43304    410  
       type_specific_thresholds  320    6  35983   7731  
fold_2 fixed_0.5                   4  148    210  43825  
       global_threshold          137   15  23180  20855  
       type_specific_thresholds  121   31  16912  27123  
fold_3 fixed_0.5                   3   36     50  43764  
       global_threshold           39    0  39596   4218  
       type_specific_thresholds   39    0  36488   7326

threshold  positive_samples    pr_auc    recall  \
fold   inspection_type                                                    
fold_1 0                 0.002325                50  0.021631  0.940000   
       1                 0.001276               186  0.216196  0.983871   
       2                 0.000020                49  0.322044  1.000000   
       3                 0.000354                39  0.524291  1.000000   
       4                 0.002461                 2  0.006154  1.000000   
fold_2 0                 0.000887                14  0.004143  0.642857   
       1                 0.000405                80  0.269666  1.000000   
       2                 0.000937                32  0.028024  0.750000   
       3                 0.006896                23  0.001672  0.217391   
       4                 0.003340                 3  0.004298  1.000000   
fold_3 0                 0.000050                 4  0.007153  1.000000   
       1                 0.000490                25  0.031508  1.000000   
       2                 0.000083                 7  0.021714  1.000000   
       3                 0.000198                 3  0.339025  1.000000   
       4                 0.003684                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
fold   inspection_type                                 
fold_1 0                            0.682749   47   3  
       1                            0.222451  183   3  
       2                            0.006714   49   0  
       3                            0.170663   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.765912    9   5  
       1                            0.344528   80   0  
       2                            0.403126   24   8  
       3                            0.725777    5  18  
       4                            0.000000    3   0  
fold_3 0                            0.045856    4   0  
       1                            0.074979   25   0  
       2                            0.043410    7   0  
       3                            0.458721    3   0  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.062963,0.080425,0.026316,0,0.996901,0.995231,52,465
global_threshold,3,0.062963,0.967105,0.901316,2,0.193083,0.009379,502,15
type_specific_thresholds,3,0.062963,0.925883,0.796053,1,0.320001,0.167207,480,37


train_rows  candidate_features  constant_removed  \
fold   inspection_type                                                     
fold_1 0                     28277                  48                14   
       1                     22698                  56                28   
       2                     42288                  69                45   
       3                     37264                  69                48   
       4                      1610                  25                 8   
fold_2 0                     36685                  48                 8   
       1                     26566                  56                28   
       2                     58736                  69                45   
       3                     51683                  69                48   
       4                      2446                  25                 8   
fold_3 0                     43181                  48                 8   
       1                     29184                  56                28   
       2                     77700                  69                45   
       3                     67320                  69                47   
       4                      2771                  25                 8   

                        duplicate_removed  residual_pairs  model_features  \
fold   inspection_type                                                      
fold_1 0                                0               7              48   
       1                                2               5              36   
       2                                0               1              26   
       3                                0               0              21   
       4                                2               0              15   
fold_2 0                                1               6              51   
       1                                2               6              38   
       2                                0               1              26   
       3                                0               0              21   
       4                                2               0              15   
fold_3 0                                0               6              52   
       1                                2               6              38   
       2                                0               1              26   
       3                                0               0              22   
       4                                2               0              15   

                        encoded_features  
fold   inspection_type                    
fold_1 0                              80  
       1                              86  
       2                              71  
       3                              59  
       4                              37  
fold_2 0                              85  
       1                              92  
       2                              71  
       3                              59  
       4                              37  
fold_3 0                              88  
       1                              93  
       2                              72  
       3                              61  
       4                              40

2026-08-25 20:37:42,895 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.06296259549754714, 'mean_recall': 0.08042522540423735, 'min_recall': 0.02631578947368421, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9969014118019054, 'min_false_call_reduction': 0.995231066197343, 'total_tp': 52, 'total_fn': 465}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.06296259549754714, 'mean_recall': 0.9671052631578947, 'min_recall': 0.9013157894736842, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.193083429909597, 'min_false_call_reduction': 0.009379146268929862, 'total_tp': 502, 'total_fn': 15}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.06296259549754714, 'mean_recall': 0.9258825745344957, 'min_recall': 0.7960526315789473, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.3200009301263706, 'min_false_call_reduction': 0.16720682886748528, 'total_tp': 480, 'total_fn': 37}}


## 8. 0~70% Train 학습과 70~80% Validation

동일한 Train-only 처리로 타입별 모델 5개를 학습합니다.

In [9]:
pooled_probability = pd.Series(np.nan, index=validation_df.index, dtype=float)
models_by_type, preprocessors_by_type, recipes_by_type = {}, {}, {}
type_metric_rows, training_rows = [], []
for inspection_type in inspection_types:
    candidates = feature_columns_by_type[inspection_type]
    train = train_df.loc[train_df[TYPE_COLUMN] == inspection_type]
    valid = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    recipe = fit_feature_recipe(train, candidates)
    train_x, valid_x = transform_recipe(train, recipe), transform_recipe(valid, recipe)
    prep = make_preprocessor(recipe["model_columns"])
    X_train, X_valid = prep.fit_transform(train_x), prep.transform(valid_x)
    model = XGBClassifier(**XGB_PARAMS).fit(X_train, train[TARGET].astype("int8"), verbose=False)
    probability = model.predict_proba(X_valid)[:, 1]; pooled_probability.loc[valid.index] = probability
    metrics = evaluate_probabilities(valid[TARGET], probability); metrics["inspection_type"] = inspection_type
    type_metric_rows.append(metrics)
    row = {"inspection_type": inspection_type, "train_rows": len(train),
           "train_positive": int(train[TARGET].sum()), "validation_rows": len(valid),
           "validation_positive": int(valid[TARGET].sum()), "candidate_features": len(candidates),
           "constant_removed": len(recipe["constants"]), "duplicate_removed": len(recipe["duplicates"]),
           "residual_pairs": len(recipe["residuals"]), "model_features": len(recipe["model_columns"]),
           "encoded_features": X_train.shape[1], "trees": model.n_estimators}
    training_rows.append(row); models_by_type[inspection_type] = model
    preprocessors_by_type[inspection_type] = prep; recipes_by_type[inspection_type] = recipe
    logger.info("final_recipe summary=%s constants=%s duplicates=%s residuals=%s",
                row, recipe["constants"], recipe["duplicates"], recipe["residuals"])
    del X_train, X_valid, train_x, valid_x, probability; gc.collect()
assert pooled_probability.notna().all()
pooled_metrics = pd.Series(evaluate_probabilities(validation_df[TARGET], pooled_probability),
                           name="type_expert_validation")
type_metrics = pd.DataFrame(type_metric_rows).set_index("inspection_type")
training_summary = pd.DataFrame(training_rows).set_index("inspection_type")
logger.info("pooled_validation_metrics=%s", pooled_metrics.to_dict())

2026-08-25 20:37:43,884 | INFO | final_recipe summary={'inspection_type': 0, 'train_rows': 64273, 'train_positive': 111, 'validation_rows': 13289, 'validation_positive': 12, 'candidate_features': 48, 'constant_removed': 7, 'duplicate_removed': 0, 'residual_pairs': 6, 'model_features': 53, 'encoded_features': 93, 'trees': 400} constants=['inspection_feat1', 'inspection_feat2', 'inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat19', 'inspection_feat22'] duplicates={} residuals=[{'left': 'inspection_feat40', 'right': 'inspection_feat53', 'block_rhos': [0.9758775195539744, 0.9979568331826008, 0.9850761211058295], 'min_abs_rho': 0.9758775195539744, 'slope': 1.1906064680273978, 'intercept': 0.006107667884886522, 'residual': 'corr_resid_00', 'absolute': 'corr_abs_resid_00'}, {'left': 'inspection_feat29', 'right': 'inspection_feat56', 'block_rhos': [0.9635182904001844, 0.9819507645663813, 0.9908194999902352], 'min_abs_rho': 0.9635182904001844, 'slope': 0.75116211607

2026-08-25 20:37:44,519 | INFO | final_recipe summary={'inspection_type': 1, 'train_rows': 38900, 'train_positive': 580, 'validation_rows': 6422, 'validation_positive': 224, 'candidate_features': 56, 'constant_removed': 28, 'duplicate_removed': 2, 'residual_pairs': 5, 'model_features': 36, 'encoded_features': 93, 'trees': 400} constants=['inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat20', 'inspection_feat21', 'inspection_feat22', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection_feat58', 'inspection_feat59', 'inspection_feat71', 'inspection_feat72', 'inspection_feat78', 'inspection_feat93', 'inspection_feat94', 'inspection_feat95', 'inspection_feat96'] duplicates={'inspection_feat32': 'inspection_feat5', 'inspection_feat33'

2026-08-25 20:37:45,578 | INFO | final_recipe summary={'inspection_type': 2, 'train_rows': 100470, 'train_positive': 588, 'validation_rows': 7161, 'validation_positive': 27, 'candidate_features': 69, 'constant_removed': 45, 'duplicate_removed': 0, 'residual_pairs': 1, 'model_features': 26, 'encoded_features': 74, 'trees': 400} constants=['inspection_feat2', 'inspection_feat6', 'inspection_feat13', 'inspection_feat16', 'inspection_feat17', 'inspection_feat21', 'inspection_feat23', 'inspection_feat24', 'inspection_feat25', 'inspection_feat26', 'inspection_feat27', 'inspection_feat29', 'inspection_feat30', 'inspection_feat31', 'inspection_feat32', 'inspection_feat33', 'inspection_feat34', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat48', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat55', 'inspection_feat56', 'inspection

2026-08-25 20:37:46,650 | INFO | final_recipe summary={'inspection_type': 3, 'train_rows': 100740, 'train_positive': 648, 'validation_rows': 16252, 'validation_positive': 21, 'candidate_features': 69, 'constant_removed': 47, 'duplicate_removed': 0, 'residual_pairs': 0, 'model_features': 22, 'encoded_features': 62, 'trees': 400} constants=['meta_feat3', 'inspection_feat2', 'inspection_feat6', 'inspection_feat13', 'inspection_feat16', 'inspection_feat18', 'inspection_feat19', 'inspection_feat21', 'inspection_feat23', 'inspection_feat24', 'inspection_feat25', 'inspection_feat26', 'inspection_feat27', 'inspection_feat29', 'inspection_feat30', 'inspection_feat31', 'inspection_feat32', 'inspection_feat33', 'inspection_feat34', 'inspection_feat40', 'inspection_feat41', 'inspection_feat42', 'inspection_feat43', 'inspection_feat44', 'inspection_feat45', 'inspection_feat46', 'inspection_feat47', 'inspection_feat48', 'inspection_feat49', 'inspection_feat53', 'inspection_feat54', 'inspection_feat5

2026-08-25 20:37:46,730 | INFO | final_recipe summary={'inspection_type': 4, 'train_rows': 3813, 'train_positive': 13, 'validation_rows': 902, 'validation_positive': 73, 'candidate_features': 25, 'constant_removed': 8, 'duplicate_removed': 2, 'residual_pairs': 0, 'model_features': 15, 'encoded_features': 43, 'trees': 400} constants=['inspection_feat16', 'inspection_feat17', 'inspection_feat18', 'inspection_feat20', 'inspection_feat21', 'inspection_feat22', 'inspection_feat23', 'inspection_feat48'] duplicates={'inspection_feat32': 'inspection_feat5', 'inspection_feat33': 'inspection_feat6'} residuals=[]


2026-08-25 20:37:46,787 | INFO | pooled_validation_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43496.0, 'fp': 173.0, 'fn': 207.0, 'tp': 150.0, 'accuracy': 0.9913687366556126, 'precision': 0.46439628482972134, 'recall': 0.42016806722689076, 'false_call_reduction': 0.9960383796285694, 'f1': 0.4411764705882353, 'roc_auc': 0.9480962368230629, 'pr_auc': 0.4379078898434957}


## 9. 최종 Validation 결과


In [10]:
count_columns = ["rows", "positive_samples", "tn", "fp", "fn", "tp"]
type_metrics[count_columns] = type_metrics[count_columns].astype("int64")
display(pooled_metrics)
display(
    type_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(training_summary)


rows                    44026.000000
positive_samples          357.000000
tn                      43496.000000
fp                        173.000000
fn                        207.000000
tp                        150.000000
accuracy                    0.991369
precision                   0.464396
recall                      0.420168
false_call_reduction        0.996038
f1                          0.441176
roc_auc                     0.948096
pr_auc                      0.437908
Name: type_expert_validation, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,13289,12,0.004693,0.853544,0.999097,0.000000,0.000000,1.000000,0.000000,0,12,0,13277
1,6422,224,0.644515,0.959772,0.964186,0.489726,0.638393,0.975960,0.554264,143,81,149,6049
2,7161,27,0.353539,0.931873,0.996788,0.750000,0.222222,0.999720,0.342857,6,21,2,7132
3,16252,21,0.073301,0.950938,0.997416,0.043478,0.047619,0.998645,0.045455,1,20,22,16209
4,902,73,0.080931,0.500000,0.919069,0.000000,0.000000,1.000000,0.000000,0,73,0,829


,train_rows,train_positive,validation_rows,validation_positive,candidate_features,constant_removed,duplicate_removed,residual_pairs,model_features,encoded_features,trees
inspection_type,,,,,,,,,,,
0,64273,111,13289,12,48,7,0,6,53,93,400
1,38900,580,6422,224,56,28,2,5,36,93,400
2,100470,588,7161,27,69,45,0,1,26,74,400
3,100740,648,16252,21,69,47,0,0,22,62,400
4,3813,13,902,73,25,8,2,0,15,43,400


## 10. 최종 Validation에서 공통·타입별 임계값 선택

Test를 사용하지 않고 Validation Recall 99% 이상을 만족하는 후보 중 False Call Reduction이 최대인 임계값을 선택합니다. 동률이면 Recall, 다시 동률이면 threshold가 높은 후보를 선택합니다.

In [11]:
global_threshold_selection = select_threshold(
    validation_df[TARGET], pooled_probability, min_recall=MIN_RECALL
)

thresholds_by_type = {}
type_threshold_rows = []
type_validation_prediction = pd.Series(np.nan, index=validation_df.index, dtype="float64")

for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_probability = pooled_probability.loc[type_validation.index]
    selection = select_threshold(
        type_validation[TARGET], type_probability, min_recall=MIN_RECALL
    )
    thresholds_by_type[inspection_type] = selection["threshold"]
    selection["inspection_type"] = inspection_type
    type_threshold_rows.append(selection)
    type_validation_prediction.loc[type_validation.index] = (
        type_probability >= selection["threshold"]
    ).astype("int8")

type_threshold_selection = pd.DataFrame(type_threshold_rows).set_index("inspection_type")
global_validation_metrics = pd.Series(
    evaluate_probabilities(
        validation_df[TARGET],
        pooled_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_specific_validation_metrics = pd.Series(
    evaluate_predictions(
        validation_df[TARGET],
        type_validation_prediction,
        pooled_probability,
    ),
    name="type_specific_thresholds",
)

validation_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": pooled_metrics,
        "global_threshold": global_validation_metrics,
        "type_specific_thresholds": type_specific_validation_metrics,
    }
).T

threshold_summary = pd.concat(
    [
        pd.DataFrame(
            [{"scope": "global", **global_threshold_selection}]
        ).set_index("scope"),
        type_threshold_selection.rename_axis("scope"),
    ],
    axis=0,
)

display(
    threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(
    validation_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
logger.info("global_threshold_selection=%s", global_threshold_selection)
logger.info("type_threshold_selection=%s", type_threshold_selection.to_dict(orient="index"))
logger.info("validation_strategy_metrics=%s", validation_strategy_metrics.to_dict(orient="index"))

,threshold,positive_samples,recall,false_call_reduction,tp,fn,fp,tn
scope,,,,,,,,
global,0.000672,357,0.991597,0.674002,354,3,14236,29433
0,0.000154,12,1.000000,0.384650,12,0,8170,5107
1,0.001154,224,0.991071,0.494837,222,2,3131,3067
2,0.000197,27,1.000000,0.433558,27,0,4041,3093
3,0.000713,21,1.000000,0.710615,21,0,4697,11534
4,0.003448,73,1.000000,0.000000,73,0,829,0


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.437908,0.464396,0.420168,0.996038,0.441176,150.0,207.0,173.0,43496.0
global_threshold,0.437908,0.024263,0.991597,0.674002,0.047367,354.0,3.0,14236.0,29433.0
type_specific_thresholds,0.437908,0.016727,0.994398,0.522132,0.032901,355.0,2.0,20868.0,22801.0


2026-08-25 20:37:46,981 | INFO | global_threshold_selection={'threshold': 0.0006715072086080909, 'min_recall': 0.99, 'rows': 44026, 'positive_samples': 357, 'tn': 29433, 'fp': 14236, 'fn': 3, 'tp': 354, 'accuracy': 0.6765774769454413, 'precision': 0.024263193968471555, 'recall': 0.9915966386554622, 'false_call_reduction': 0.6740021525567336, 'f1': 0.04736736468856627, 'roc_auc': 0.9480962368230629, 'pr_auc': 0.4379078898434957}


2026-08-25 20:37:46,982 | INFO | type_threshold_selection={0: {'threshold': 0.00015380022523459047, 'min_recall': 0.99, 'rows': 13289, 'positive_samples': 12, 'tn': 5107, 'fp': 8170, 'fn': 0, 'tp': 12, 'accuracy': 0.38520580931597564, 'precision': 0.0014666340747983377, 'recall': 1.0, 'false_call_reduction': 0.38465014687052795, 'f1': 0.002928972418843056, 'roc_auc': 0.8535437222264065, 'pr_auc': 0.004693314446250095}, 1: {'threshold': 0.0011537419632077217, 'min_recall': 0.99, 'rows': 6422, 'positive_samples': 224, 'tn': 3067, 'fp': 3131, 'fn': 2, 'tp': 222, 'accuracy': 0.5121457489878543, 'precision': 0.06620936474798687, 'recall': 0.9910714285714286, 'false_call_reduction': 0.494837044207809, 'f1': 0.12412636287391669, 'roc_auc': 0.9597724496381321, 'pr_auc': 0.6445146190693726}, 2: {'threshold': 0.00019721586431842297, 'min_recall': 0.99, 'rows': 7161, 'positive_samples': 27, 'tn': 3093, 'fp': 4041, 'fn': 0, 'tp': 27, 'accuracy': 0.43569333891914536, 'precision': 0.0066371681415929

2026-08-25 20:37:46,983 | INFO | validation_strategy_metrics={'fixed_0.5': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43496.0, 'fp': 173.0, 'fn': 207.0, 'tp': 150.0, 'accuracy': 0.9913687366556126, 'precision': 0.46439628482972134, 'recall': 0.42016806722689076, 'false_call_reduction': 0.9960383796285694, 'f1': 0.4411764705882353, 'roc_auc': 0.9480962368230629, 'pr_auc': 0.4379078898434957}, 'global_threshold': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 29433.0, 'fp': 14236.0, 'fn': 3.0, 'tp': 354.0, 'accuracy': 0.6765774769454413, 'precision': 0.024263193968471555, 'recall': 0.9915966386554622, 'false_call_reduction': 0.6740021525567336, 'f1': 0.04736736468856627, 'roc_auc': 0.9480962368230629, 'pr_auc': 0.4379078898434957}, 'type_specific_thresholds': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 22801.0, 'fp': 20868.0, 'fn': 2.0, 'tp': 355.0, 'accuracy': 0.5259619315858811, 'precision': 0.016727135654714224, 'recall': 0.9943977591036415, 'false_call_reduction': 

## 11. 고정 모델의 진단용 Test 평가

Validation에서 모델과 임계값을 고정한 뒤 마지막 20% Test를 한 번 평가합니다. 이 결과를 사용한 추가 튜닝이나 후보 선택은 하지 않습니다.


In [12]:
test_probability = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_test_metric_rows = []

for inspection_type in inspection_types:
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    recipe = recipes_by_type[inspection_type]
    test_features = transform_recipe(type_test, recipe)
    X_test = preprocessors_by_type[inspection_type].transform(test_features)
    probability = models_by_type[inspection_type].predict_proba(X_test)[:, 1]
    test_probability.loc[type_test.index] = probability
    metrics = evaluate_probabilities(type_test[TARGET], probability)
    metrics["inspection_type"] = inspection_type
    type_test_metric_rows.append(metrics)
    del X_test, test_features, probability
    gc.collect()

assert test_probability.notna().all()
fixed_test_metrics = pd.Series(
    evaluate_probabilities(test_df[TARGET], test_probability), name="fixed_0.5"
)
global_test_metrics = pd.Series(
    evaluate_probabilities(
        test_df[TARGET], test_probability, global_threshold_selection["threshold"]
    ),
    name="global_threshold",
)
type_test_prediction = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_selected_test_rows = []
for inspection_type in inspection_types:
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    probability = test_probability.loc[type_test.index]
    threshold = thresholds_by_type[inspection_type]
    prediction = (probability >= threshold).astype("int8")
    type_test_prediction.loc[type_test.index] = prediction
    metrics = evaluate_predictions(type_test[TARGET], prediction, probability)
    metrics.update({"inspection_type": inspection_type, "threshold": threshold})
    type_selected_test_rows.append(metrics)

type_specific_test_metrics = pd.Series(
    evaluate_predictions(test_df[TARGET], type_test_prediction, test_probability),
    name="type_specific_thresholds",
)
test_strategy_metrics = pd.DataFrame(
    {"fixed_0.5": fixed_test_metrics,
     "global_threshold": global_test_metrics,
     "type_specific_thresholds": type_specific_test_metrics}
).T
type_test_metrics = pd.DataFrame(type_test_metric_rows).set_index("inspection_type")
type_selected_test_metrics = pd.DataFrame(type_selected_test_rows).set_index("inspection_type")

display(test_strategy_metrics[
    ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
])
display(type_selected_test_metrics[
    ["threshold", "positive_samples", "pr_auc", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
])
logger.info("test_strategy_metrics=%s", test_strategy_metrics.to_dict(orient="index"))
logger.info("test_evaluation_policy=diagnostic_only no_post_test_tuning=True")


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.329307,0.432296,0.221075,0.992126,0.292544,514.0,1811.0,675.0,85052.0
global_threshold,0.329307,0.054564,0.899785,0.577169,0.102889,2092.0,233.0,36248.0,49479.0
type_specific_thresholds,0.329307,0.042878,0.926882,0.438870,0.081964,2155.0,170.0,48104.0,37623.0


,threshold,positive_samples,pr_auc,recall,false_call_reduction,tp,fn,fp,tn
inspection_type,,,,,,,,,
0,0.000154,195,0.034163,0.830769,0.385313,162,33,11861,7435
1,0.001154,774,0.384331,0.978036,0.458236,757,17,6272,5305
2,0.000197,731,0.627997,0.949384,0.235312,694,37,15150,4662
3,0.000713,612,0.356816,0.864379,0.589070,529,83,14106,20221
4,0.003448,13,0.017857,1.000000,0.000000,13,0,715,0


2026-08-25 20:37:47,751 | INFO | test_strategy_metrics={'fixed_0.5': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 85052.0, 'fp': 675.0, 'fn': 1811.0, 'tp': 514.0, 'accuracy': 0.9717666833234907, 'precision': 0.432296047098402, 'recall': 0.2210752688172043, 'false_call_reduction': 0.9921261679517538, 'f1': 0.292544109277177, 'roc_auc': 0.8751723318747147, 'pr_auc': 0.32930670055365635}, 'global_threshold': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 49479.0, 'fp': 36248.0, 'fn': 233.0, 'tp': 2092.0, 'accuracy': 0.5856880025439513, 'precision': 0.05456442357850808, 'recall': 0.8997849462365591, 'false_call_reduction': 0.5771693865409964, 'f1': 0.10288946268289684, 'roc_auc': 0.8751723318747147, 'pr_auc': 0.32930670055365635}, 'type_specific_thresholds': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 37623.0, 'fp': 48104.0, 'fn': 170.0, 'tp': 2155.0, 'accuracy': 0.4517557806750557, 'precision': 0.04287789251676317, 'recall': 0.9268817204301075, 'false_call_reduction': 

2026-08-25 20:37:47,752 | INFO | test_evaluation_policy=diagnostic_only no_post_test_tuning=True


## 12. 원본 무결성과 종료 확인


In [13]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE
assert len(models_by_type) == 5 and test_probability.notna().all()
verification = pd.Series(
    {"dataset_sha256_unchanged": True,
     "mapping_sha256_unchanged": True,
     "type_models_trained": 5,
     "test_evaluated_once": True,
     "post_test_tuning_allowed": False,
     "global_validation_threshold": global_threshold_selection["threshold"],
     "log_file": f"docs/peace/{LOG_PATH.name}"},
    name="verification",
)
display(verification)
logger.info("source_integrity=PASS test_evaluated_once=True post_test_tuning_allowed=False")
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers: handler.flush()


dataset_sha256_unchanged                                                    True
mapping_sha256_unchanged                                                    True
type_models_trained                                                            5
test_evaluated_once                                                         True
post_test_tuning_allowed                                                   False
global_validation_threshold                                             0.000672
log_file                       docs/peace/0825_peace_020_type_expert_preproce...
Name: verification, dtype: object

2026-08-25 20:37:47,938 | INFO | source_integrity=PASS test_evaluated_once=True post_test_tuning_allowed=False


2026-08-25 20:37:47,939 | INFO | experiment_complete=0825_peace_020_type_expert_preprocessing_correlation_residual


## 13. Result

Residual features raised Test PR-AUC to 0.329307 but did not improve the operating target. The global Validation threshold produced Recall 89.98% and FCR 57.72%, below the 004 FCR. Residual features are rejected, and this Test result must not be used for further tuning.
